# fraud detection on kaggle call transcripts

basic nlp pipeline to classify scam vs legit calls

In [17]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack
import numpy as np

## load data

In [18]:
df_original = pd.read_csv("KaggleCallTranscripts.csv")
print(f"Original shape: {df_original.shape}")
df_original.head()

Original shape: (650, 7)


,CONVERSATION_ID,CONVERSATION_STEP,TEXT,CONTEXT,LABEL,FEATURES,ANNOTATIONS
0,6,1,"Good morning, this is [Your Name]'s personal a...",Standard opening exchange,neutral,NaN,NaN
1,6,2,"Hello, my name is Jamie. I'm interested in vol...",Encourages the caller's interest,neutral,"welcoming, positive_tone",NaN
2,6,3,"Yes, I'm really passionate about environmental...",Reinforces anyone can volunteer,neutral,inclusive,NaN
3,6,4,"Great, how do I sign up, and where can I find ...",Demonstrates flexibility,neutral,"helpful_tone, offers_options",NaN
4,6,5,"Could you send me the link, please? And my ema...",Fulfills caller's request quickly,neutral,prompt_action,NaN


In [19]:
# labels are a mess - 20+ different values
df_original["LABEL"].value_counts()

LABEL
neutral                                     158
 scam                                       138
 scam_response                              117
scam                                         82
legitimate                                   43
 neutral                                     35
suspicious                                   32
 legitimate                                  14
slightly_suspicious                           8
potential_scam                                7
highly_suspicious                             3
scam_response                                 3
 citing urgency"                              2
polite_ending                                 1
standard_opening, identification_request      1
 dismissing official protocols"               1
 emphasizing security and compliance"         1
 ready for further engagement"                1
 suggesting a dangerous situation"            1
Scam                                          1
 adhering to protocols"           

## clean the messy labels

converting 20+ label variations into just scam/legitimate

In [20]:
df_cleaned = df_original.copy()

# anything with scam/suspicious/fraud keywords = scam, rest = legitimate
scam_keywords = ['scam', 'suspicious', 'fraud', 'urgency', 'threat']

def clean_label(label):
    if pd.isna(label):
        return 'legitimate'
    
    label_lower = str(label).lower()
    if any(keyword in label_lower for keyword in scam_keywords):
        return 'scam'
    else:
        return 'legitimate'

df_cleaned['LABEL_BINARY'] = df_cleaned['LABEL'].apply(clean_label)
df_cleaned['LABEL_BINARY'].value_counts()

LABEL_BINARY
scam          393
legitimate    257
Name: count, dtype: int64

## extract custom features

trying 2 features:
- urgency words (now, immediately, urgent, etc)  
- verification requests (asking for SSN, account #, password)

In [21]:
def count_urgency_words(text):
    urgency_words = ['immediately', 'urgent', 'now', 'today', 'expires', 'hurry', 
                     'asap', 'quickly', 'deadline', 'limited time']
    text_lower = text.lower()
    return sum(1 for word in urgency_words if word in text_lower)

def count_verification_requests(text):
    # patterns that ask for personal info
    verification_patterns = [
        r'verify\s+your', r'confirm\s+your', r'provide\s+your',
        r'enter\s+your', r'social\s+security', r'account\s+number',
        r'password', r'credit\s+card', r'bank\s+account'
    ]
    return sum(1 for pattern in verification_patterns if re.search(pattern, text.lower()))

df_cleaned['urgency_score'] = df_cleaned['TEXT'].apply(count_urgency_words)
df_cleaned['verification_requests'] = df_cleaned['TEXT'].apply(count_verification_requests)

In [22]:
# check if these features differ between scam and legit
print(f"avg urgency - scam: {df_cleaned[df_cleaned['LABEL_BINARY']=='scam']['urgency_score'].mean():.2f}")
print(f"avg urgency - legit: {df_cleaned[df_cleaned['LABEL_BINARY']=='legitimate']['urgency_score'].mean():.2f}")
print(f"\navg verification requests - scam: {df_cleaned[df_cleaned['LABEL_BINARY']=='scam']['verification_requests'].mean():.2f}")
print(f"avg verification requests - legit: {df_cleaned[df_cleaned['LABEL_BINARY']=='legitimate']['verification_requests'].mean():.2f}")

avg urgency - scam: 0.19
avg urgency - legit: 0.32

avg verification requests - scam: 0.06
avg verification requests - legit: 0.02


In [23]:
df_cleaned.to_csv("KaggleCallTranscripts_cleaned.csv", index=False)

## baseline model - just text

tfidf = converts text to numbers  
basically finds words that are distinctive for scam vs legit (like "verify" appears a lot in scams but not legit)

In [ ]:
X = df_cleaned["TEXT"]
y = df_cleaned["LABEL_BINARY"]

# 80/20 split, stratify keeps same scam/legit ratio in train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"train: {len(X_train)}, test: {len(X_test)}")

Training set: 520 samples
Test set: 130 samples


In [ ]:
# convert text to numbers using tfidf
vectorizer = TfidfVectorizer(
    stop_words='english',  # ignore common words like "the", "a"
    max_features=1000,     # only keep top 1000 most important words
    ngram_range=(1, 2)     # use single words and 2-word phrases
)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model_baseline = LogisticRegression(max_iter=1000, random_state=42)
model_baseline.fit(X_train_tfidf, y_train)

y_pred_baseline = model_baseline.predict(X_test_tfidf)
accuracy_baseline = (y_pred_baseline == y_test).mean()

print(f"baseline accuracy: {accuracy_baseline:.2%}")

Baseline accuracy: 95.38%


## enhanced model - text + custom features

adding our 2 manual features to see if it helps

In [ ]:
feature_cols = ['urgency_score', 'verification_requests']
df_train = df_cleaned.loc[X_train.index]
df_test = df_cleaned.loc[X_test.index]

custom_features_train = df_train[feature_cols].values
custom_features_test = df_test[feature_cols].values

# scale features so they're on similar scale as tfidf
scaler = StandardScaler()
custom_features_train_scaled = scaler.fit_transform(custom_features_train)
custom_features_test_scaled = scaler.transform(custom_features_test)

# combine tfidf + custom features
X_train_combined = hstack([X_train_tfidf, custom_features_train_scaled])
X_test_combined = hstack([X_test_tfidf, custom_features_test_scaled])

model_enhanced = LogisticRegression(max_iter=1000, random_state=42)
model_enhanced.fit(X_train_combined, y_train)

y_pred_enhanced = model_enhanced.predict(X_test_combined)
accuracy_enhanced = (y_pred_enhanced == y_test).mean()

print(f"enhanced accuracy: {accuracy_enhanced:.2%}")
print(f"improvement: {(accuracy_enhanced - accuracy_baseline)*100:.2f} points")

Enhanced accuracy: 96.15%
Improvement: 0.77 percentage points


## results

In [ ]:
print(classification_report(y_test, y_pred_enhanced))

Classification Report:
              precision    recall  f1-score   support

  legitimate       0.98      0.92      0.95        51
        scam       0.95      0.99      0.97        79

    accuracy                           0.96       130
   macro avg       0.97      0.95      0.96       130
weighted avg       0.96      0.96      0.96       130



In [ ]:
cm = confusion_matrix(y_test, y_pred_enhanced)
print(cm)
print("\n[legit predicted legit, legit predicted scam]")
print("[scam predicted legit, scam predicted scam]")

Confusion Matrix:
[[47  4]
 [ 1 78]]

[True Negatives, False Positives]
[False Negatives, True Positives]


## what words matter most

In [ ]:
feature_names = vectorizer.get_feature_names_out()
coefficients = model_enhanced.coef_[0][:-2]  # exclude the 2 custom features

scam_indices = np.argsort(coefficients)[-10:]
print("top scam words:")
for idx in reversed(scam_indices):
    print(f"  {feature_names[idx]}: {coefficients[idx]:.3f}")

Top 10 Scam Indicators:
  need: 1.270
  official: 1.090
  proceed: 0.963
  verify: 0.945
  security: 0.902
  immediate: 0.776
  data: 0.717
  grandson: 0.695
  provide details: 0.683
  investment: 0.678


In [ ]:
legit_indices = np.argsort(coefficients)[:10]
print("top legit words:")
for idx in legit_indices:
    print(f"  {feature_names[idx]}: {coefficients[idx]:.3f}")

Top 10 Legitimate Indicators:
  step: -2.910
  assistant: -1.649
  assist: -1.499
  event: -1.472
  thank: -1.465
  great: -1.358
  email: -1.303
  today: -1.295
  hello: -1.278
  assistant assist: -1.162


In [ ]:
# how much do our custom features help?
custom_coefs = model_enhanced.coef_[0][-2:]
print("custom features:")
print(f"  urgency: {custom_coefs[0]:.3f}")
print(f"  verification requests: {custom_coefs[1]:.3f}")

Custom Feature Coefficients:
  urgency_score: -0.033
  verification_requests: 0.074


In [ ]:
# reality check - tiny dataset
print(f"only {len(df_cleaned)} conversations total")
print("model probably overfitting, won't generalize well")


Dataset size: 650 conversations
Note: This is a small dataset and may overfit.
